# 🧠 Credit Card Customer Churn Prediction

## 📌 Project Overview

Customer churn is a major challenge for financial institutions. Retaining existing customers is often more cost-effective than acquiring new ones.

This project builds a **machine learning model to predict whether a customer will leave the bank (churn)** based on demographic and financial attributes.

The final model is deployed using a **Streamlit dashboard** that allows users to input customer details and receive churn predictions in real time.

## 📂 Dataset Description

The dataset used in this project is the **Credit Card Customer Churn dataset from Kaggle**.

- **Total Records:** 10,000 customers  
- **Features:** 14 columns (RowNumber, CustomerId, Surname dropped before modelling)
- **Target Variable:** `Exited` (1 = Churned, 0 = Stayed)

### 🔑 Key Features

| Feature | Description |
|------|------|
| CreditScore | Customer credit score |
| Geography | Customer location |
| Gender | Male/Female |
| Age | Customer age |
| Tenure | Years with bank |
| Balance | Account balance |
| NumOfProducts | Number of bank products |
| HasCrCard | Has credit card |
| IsActiveMember | Active member status |
| EstimatedSalary | Customer salary |

In [ ]:
!pip install tensorflow scikit-learn pandas numpy matplotlib seaborn joblib

In [ ]:
# ── All imports ──────────────────────────────────────────────────────────────
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# ── Load data once ───────────────────────────────────────────────────────────
df = pd.read_csv("../data/raw/Churn_Modelling.csv")

print("Data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

## 📊 Exploratory Data Analysis (EDA)

Exploratory Data Analysis (EDA) helps understand the dataset and identify patterns that may influence customer churn.  
In this section, we analyze how different factors such as customer demographics and account information relate to churn behavior.

> **Note:** EDA is performed on the raw dataframe before any preprocessing. All transformations (dropping columns, encoding) happen later in the Preprocessing section.

---

### 📉 Churn Distribution

This visualization shows the distribution of customers who **left the bank (churned)** versus those who **remained**.

---

### 🌍 Geographic Impact on Churn

This analysis explores how **customer location (Geography)** affects churn behavior.

---

### 👥 Age-Based Churn Patterns

Customer age can play an important role in predicting churn.

In [ ]:
# ── EDA Plot 1: Churn distribution ───────────────────────────────────────────
# EDA uses the raw df without any modifications
churn_counts = df['Exited'].value_counts()
plt.figure(figsize=(6, 6))
plt.pie(churn_counts, labels=['Stayed', 'Exited'], autopct='%1.1f%%',
        colors=['#2ecc71', '#e74c3c'])
plt.title('Churn Distribution')
plt.show()

print(f"Stayed : {churn_counts[0]:,} ({churn_counts[0]/len(df)*100:.1f}%)")
print(f"Churned: {churn_counts[1]:,} ({churn_counts[1]/len(df)*100:.1f}%)")

The dataset shows an imbalanced class distribution:
- **79.6%** of customers stayed with the bank
- **20.4%** of customers churned (left the bank)

This imbalance (approximately 80/20) is typical for churn prediction problems and is addressed during model training using **class weights**.

In [ ]:
# ── EDA Plot 2: Geography vs churn ──────────────────────────────────────────
# Work on a temporary copy so the raw df stays untouched
df_geo = df.copy()
df_geo = pd.get_dummies(df_geo, columns=['Geography'], drop_first=False)

germany_churn     = df_geo[df_geo['Geography_Germany'] == 1]['Exited'].mean() * 100
non_germany_churn = df_geo[df_geo['Geography_Germany'] == 0]['Exited'].mean() * 100

plt.figure(figsize=(8, 5))
categories  = ['Non-Germany', 'Germany']
churn_rates = [non_germany_churn, germany_churn]
colors      = ['#3498db', '#e74c3c']

bars = plt.bar(categories, churn_rates, color=colors, edgecolor='black')
plt.title('Churn Rate: Germany vs Other Countries', fontsize=14, fontweight='bold')
plt.ylabel('Churn Rate (%)')
plt.ylim(0, 50)

for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., h + 1,
             f'{h:.1f}%', ha='center', fontweight='bold')
plt.show()

print(f"Germany churn rate    : {germany_churn:.1f}%")
print(f"Non-Germany churn rate: {non_germany_churn:.1f}%")

In [ ]:
# ── EDA Plot 3: Age group vs churn ──────────────────────────────────────────
df_age = df.copy()
df_age['Age_group'] = pd.cut(
    df_age['Age'],
    bins=[0, 30, 40, 50, 60, 100],
    labels=['<30', '30-40', '40-50', '50-60', '60+']
)

age_groups      = ['<30', '30-40', '40-50', '50-60', '60+']
age_churn_rates = []

for ag in age_groups:
    grp        = df_age[df_age['Age_group'] == ag]
    rate       = grp['Exited'].mean() * 100
    age_churn_rates.append(rate)
    print(f"{ag}: {rate:.1f}% churn rate ({len(grp)} customers)")

colors = ['#2ecc71', '#3498db', '#f39c12', '#e67e22', '#e74c3c']
plt.figure(figsize=(10, 6))
bars = plt.bar(age_groups, age_churn_rates, color=colors, edgecolor='black')
plt.title('Churn Rate by Age Group', fontsize=14, fontweight='bold')
plt.xlabel('Age Group')
plt.ylabel('Churn Rate (%)')
plt.ylim(0, 50)

for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., h + 1,
             f'{h:.1f}%', ha='center', fontweight='bold')
plt.show()

print(f"\n👉 Highest churn age group: {age_groups[np.argmax(age_churn_rates)]} "
      f"({max(age_churn_rates):.1f}%)")

## 🔍 Key Insights from EDA

| Insight | Business Implication |
|---------|---------------------|
| **20.4% overall churn rate** | Significant revenue at risk — retention efforts needed |
| **Germany has highest churn** | Target German customers with special retention offers |
| **Age 50-60 highest churn** | Focus retention on middle-aged customers |
| **Age <30 lowest churn** | Young customers are loyal — leverage for referrals |

## ⚙️ Data Preprocessing

Before training the model, the dataset was prepared using the following steps.

---

### 🗑 Drop Non-Informative Columns

`RowNumber`, `CustomerId`, and `Surname` carry no predictive signal and are removed.

---

### 🔤 Encoding Categorical Variables

Categorical features are converted to numerical format using **One-Hot Encoding** with `pandas.get_dummies(drop_first=True)`.  
The **encoder column list is saved** alongside the scaler so inference can produce identical columns.

Encoded columns:
- `Geography`
- `Gender`

---

### 🎯 Feature and Target Split

- **X (Features):** All input variables  
- **y (Target):** `Exited` (1 = churned, 0 = stayed)

---

### 🔀 Train–Test Split

- Training data: 80%  
- Testing data: 20%  
- Random state: 1

---

### 📏 Feature Scaling

Numerical features are standardised using `StandardScaler`.  
`fit_transform()` is applied to training data only; `transform()` is applied to test data.

In [ ]:
# ── Preprocessing ────────────────────────────────────────────────────────────

# 1. Work on a clean copy; the raw df is left intact for reference
df_model = df.copy()

# 2. Drop non-informative columns
df_model.drop(columns=['RowNumber', 'CustomerId', 'Surname'], inplace=True)

# 3. One-hot encode categorical columns — happens EXACTLY ONCE
df_model = pd.get_dummies(df_model, columns=['Geography', 'Gender'], drop_first=True)

print("Columns after encoding:", df_model.columns.tolist())

# 4. Save processed dataset
os.makedirs("../data/processed", exist_ok=True)
df_model.to_csv("../data/processed/Churn_processed.csv", index=False)
print("\nProcessed dataset saved to ../data/processed/Churn_processed.csv")

# 5. Feature / target split
X = df_model.drop(columns=['Exited'])
y = df_model['Exited']

# 6. Save the training column order — required for consistent inference later
training_columns = X.columns.tolist()
joblib.dump(training_columns, '../models/training_columns.pkl')
print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Class balance — churned: {y.mean()*100:.1f}%")

In [ ]:
# ── Train-test split ─────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

print(f"X_train: {X_train.shape}  X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}  y_test: {y_test.shape}")

# ── Feature scaling ───────────────────────────────────────────────────────────
scaler          = StandardScaler()
X_train_scaled  = scaler.fit_transform(X_train)
X_test_scaled   = scaler.transform(X_test)

print("\nScaling complete.")

## 🤖 Model Training

To identify the best performing algorithm, multiple machine learning models were evaluated.

### 🧠 Models Tested
1. Logistic Regression  
2. Random Forest  
3. Support Vector Machine (SVM)  
4. Neural Network  

### 📏 Evaluation Metrics
- **Accuracy** (headline number)  
- **Precision, Recall, F1-score** for the minority (churn) class  
- `class_weight='balanced'` is applied to all sklearn models and the Keras model to address the 80/20 class imbalance

| Model | Accuracy |
|------|------|
| Logistic Regression | 82% |
| Random Forest | 86% |
| SVM | 86% |
| Neural Network | 86% |

**The Neural Network was selected to explore deep learning approaches.** SVM achieved marginally higher raw accuracy but the neural network provides more flexibility for future improvements.

In [ ]:
# ── Baseline model comparison (with class_weight='balanced') ─────────────────

# Compute class weight ratio for Keras
neg, pos   = np.bincount(y_train)
class_weight_ratio = neg / pos   # weight to give the minority class
print(f"Class weight ratio (neg/pos): {class_weight_ratio:.2f}")

sklearn_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Random Forest"      : RandomForestClassifier(class_weight='balanced', random_state=42),
    "SVM"                : SVC(class_weight='balanced'),
}

results = {}
for name, mdl in sklearn_models.items():
    mdl.fit(X_train_scaled, y_train)
    preds        = mdl.predict(X_test_scaled)
    results[name] = accuracy_score(y_test, preds)
    print(f"{name}: {results[name]:.4f}")

# Quick Neural Network baseline for comparison
nn_baseline = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1,  activation='sigmoid')
])
nn_baseline.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
nn_baseline.fit(X_train_scaled, y_train,
                epochs=20, batch_size=32, verbose=0,
                class_weight={0: 1.0, 1: class_weight_ratio})

_, nn_acc = nn_baseline.evaluate(X_test_scaled, y_test, verbose=0)
results["Neural Network"] = nn_acc
print(f"Neural Network (baseline): {nn_acc:.4f}")

results_df = pd.DataFrame(results.items(), columns=["Model", "Accuracy"])
results_df["Accuracy"] = results_df["Accuracy"].round(4)
print("\n", results_df.to_string(index=False))

## 🧠 Neural Network Model Architecture

The deep learning model was implemented using **TensorFlow / Keras**.

### 🏗 Model Architecture

| Layer | Configuration |
|------|------|
| Input Layer | 11 features |
| Dense Layer | 64 neurons (ReLU) |
| Dropout | 0.3 |
| Dense Layer | 32 neurons (ReLU) |
| Dropout | 0.3 |
| Output Layer | 1 neuron (Sigmoid) |

### ⚡ Training Configuration
- **Loss Function:** Binary Crossentropy  
- **Optimizer:** Adam  
- **Epochs:** 50  
- **Batch Size:** 32  
- **Class weights:** Applied to address imbalance

In [ ]:
# ── Final Neural Network model ───────────────────────────────────────────────
tf.keras.backend.clear_session()

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# ── Training ─────────────────────────────────────────────────────────────────
history = model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
    class_weight={0: 1.0, 1: class_weight_ratio}
)

test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\n✅ Test Accuracy: {test_accuracy:.4f}")

## 📊 Model Evaluation

After training, the model is evaluated on the held-out test set using:
- **Accuracy** — overall correctness
- **Classification report** — precision, recall, F1 for each class
- **Confusion matrix** — breakdown of correct and incorrect predictions

In [ ]:
# ── Evaluation on test set ───────────────────────────────────────────────────
y_pred_prob = model.predict(X_test_scaled)
y_pred      = (y_pred_prob > 0.5).astype(int).flatten()

print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Stayed', 'Churned']))

## 📊 Model Performance Visualizations

### Training History Analysis

The graphs below show how the model learned over 50 epochs:

- **Loss Plot**: Shows how the error decreased during training
- **Accuracy Plot**: Shows how prediction accuracy improved

### Confusion Matrix

- **Top-left**: Customers correctly predicted to STAY
- **Bottom-right**: Customers correctly predicted to EXIT
- **Top-right**: False alarms (predicted exit but actually stayed)
- **Bottom-left**: Missed churners (predicted stay but actually exited)

### Feature Importance

Approximated via **permutation importance** — measures how much accuracy drops when each feature is randomly shuffled. This is a valid importance measure unlike raw first-layer weights.

In [ ]:
# ── Performance visualisations ───────────────────────────────────────────────
from sklearn.inspection import permutation_importance

plt.style.use('seaborn-v0_8-darkgrid')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Loss
axes[0, 0].plot(history.history['loss'],     label='Training Loss',   linewidth=2)
axes[0, 0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0, 0].set_title('Model Loss',     fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epochs')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()

# 2. Accuracy
axes[0, 1].plot(history.history['accuracy'],     label='Training Accuracy',   linewidth=2)
axes[0, 1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0, 1].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epochs')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()

# 3. Confusion matrix
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Stayed', 'Churned'])
disp.plot(ax=axes[1, 0], values_format='d')
axes[1, 0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')

# 4. Permutation importance (valid for neural networks)
# Wrap the Keras model in a sklearn-compatible class
from sklearn.base import BaseEstimator, ClassifierMixin

class KerasWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, model): self.model = model
    def fit(self, X, y): return self
    def predict(self, X): return (self.model.predict(X, verbose=0) > 0.5).astype(int).flatten()
    def score(self, X, y): return accuracy_score(y, self.predict(X))

wrapped    = KerasWrapper(model)
perm_imp   = permutation_importance(wrapped, X_test_scaled, y_test,
                                    n_repeats=10, random_state=42)
feat_names = X_train.columns.tolist()
perm_df    = pd.DataFrame({'Feature': feat_names,
                            'Importance': perm_imp.importances_mean}).sort_values('Importance')

axes[1, 1].barh(perm_df['Feature'], perm_df['Importance'])
axes[1, 1].set_title('Permutation Feature Importance', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Mean accuracy decrease')

plt.tight_layout()
plt.show()

In [ ]:
# ── Save model, scaler, and column list ──────────────────────────────────────
os.makedirs("../models", exist_ok=True)

# Save in native Keras format (recommended over legacy .h5)
model.save('../models/churn_prediction_model.keras')

# Save scaler
joblib.dump(scaler, '../models/scaler.pkl')

# Save training column list (needed at inference time to align new inputs)
joblib.dump(training_columns, '../models/training_columns.pkl')

print("Model saved to  : ../models/churn_prediction_model.keras")
print("Scaler saved to : ../models/scaler.pkl")
print("Columns saved to: ../models/training_columns.pkl")

## 🔮 Inference — Predicting Churn for New Customers

At inference time the same three assets are loaded:
1. `churn_prediction_model.keras` — the trained neural network
2. `scaler.pkl` — the fitted `StandardScaler`
3. `training_columns.pkl` — the exact column list produced after one-hot encoding

This guarantees that new customer data is encoded identically to how the training data was processed.

In [ ]:
# ── Predict new customers ────────────────────────────────────────────────────
# Load saved assets (simulates how app.py would use them)
_model   = tf.keras.models.load_model('../models/churn_prediction_model.keras')
_scaler  = joblib.load('../models/scaler.pkl')
_columns = joblib.load('../models/training_columns.pkl')

def predict_customers(customer_list):
    """
    Encode, scale, and predict churn for a list of customer dicts.
    Uses the persisted column list so encoding is always consistent
    with training — no risk of column mismatch.
    """
    df_new = pd.DataFrame(customer_list)
    df_new = df_new.drop(columns=['RowNumber', 'CustomerId', 'Surname'],
                          errors='ignore')
    df_new = pd.get_dummies(df_new, columns=['Geography', 'Gender'], drop_first=True)
    df_new = df_new.reindex(columns=_columns, fill_value=0)
    scaled = _scaler.transform(df_new)
    probs  = _model.predict(scaled, verbose=0).flatten()
    return probs

# ── Test customers ────────────────────────────────────────────────────────────
test_customers = [
    {"CreditScore": 619,  "Geography": "France",  "Gender": "Female", "Age": 42,
     "Tenure": 2,  "Balance": 0.0,    "NumOfProducts": 1, "HasCrCard": 1,
     "IsActiveMember": 1, "EstimatedSalary": 101348.88},

    {"CreditScore": 450,  "Geography": "Germany", "Gender": "Male",   "Age": 55,
     "Tenure": 1,  "Balance": 150000, "NumOfProducts": 2, "HasCrCard": 1,
     "IsActiveMember": 0, "EstimatedSalary": 50000},

    {"CreditScore": 850,  "Geography": "France",  "Gender": "Female", "Age": 30,
     "Tenure": 5,  "Balance": 0,      "NumOfProducts": 1, "HasCrCard": 1,
     "IsActiveMember": 1, "EstimatedSalary": 150000},

    {"CreditScore": 600,  "Geography": "Spain",   "Gender": "Male",   "Age": 45,
     "Tenure": 3,  "Balance": 100000, "NumOfProducts": 1, "HasCrCard": 0,
     "IsActiveMember": 0, "EstimatedSalary": 75000},
]

probs = predict_customers(test_customers)

print("=" * 50)
print("PREDICTION RESULTS")
print("=" * 50)
for i, prob in enumerate(probs):
    risk = "HIGH 🔴" if prob > 0.5 else "LOW 🟢"
    print(f"Customer {i+1}: Churn Probability = {prob:.2%}  —  {risk} RISK")

### 🏆 Final Model Selection

Several machine learning models were evaluated, including Logistic Regression, Random Forest, SVM, and a Neural Network.

The **Neural Network model built using TensorFlow/Keras** was selected as the final model for this project because it can capture complex, non-linear patterns in customer behaviour.

This trained model is deployed in a **Streamlit application** to predict whether a customer is likely to churn based on their information.

## 🚀 Model Deployment

After evaluating the model, it was deployed using **Streamlit** to create an interactive web application that allows users to predict customer churn in real time.

---

### 🖥 Deployment Framework

The application provides an interactive interface where users can input customer information and receive churn predictions instantly.

---

### 📊 Application Features

The deployed dashboard includes:

- **Customer Input Panel** — credit score, age, tenure, balance, products, active status
- **Model Information** — Neural Network, ~85% accuracy
- **Prediction System** — probability score + HIGH / LOW risk label

---

### ⚙️ Running the Application

```bash
streamlit run app/app.py
```

The application loads `churn_prediction_model.keras`, `scaler.pkl`, and `training_columns.pkl` from the `../models/` directory.

## 🎯 Final Summary

### ✅ What We Accomplished

- Built a **Neural Network model** to predict customer churn
- Achieved **~85% test accuracy** (with class-balanced training)
- Performed **data preprocessing, feature scaling, and categorical encoding**
- Trained and evaluated multiple models and selected the best performer
- Identified key churn indicators such as **Geography, Age, and Customer Activity**
- Developed an **interactive prediction dashboard using Streamlit**

---

### 💡 Business Recommendations

1. **Target German customers** with special retention offers as they show higher churn probability.
2. **Engage inactive members** through loyalty programs, personalised emails, or incentives.
3. **Focus on customers aged 40–60**, as this group demonstrates higher churn risk.
4. Introduce **personalised financial products** based on customer behaviour and risk score.

---

### 🚀 Future Improvements

- Improve model performance using **Gradient Boosting or XGBoost**
- Perform **hyperparameter tuning** to further optimise prediction accuracy
- Incorporate **additional customer behaviour features** such as transaction frequency and service usage
- Implement **A/B testing strategies** to evaluate the effectiveness of churn reduction campaigns
- Build a **real-time data pipeline** for automated churn monitoring

---

### 📋 Project Information

**Created by:** Ayushi Rai  
**Project:** Customer Churn Prediction  
**Model:** Neural Network  
**Test Accuracy:** ~85%  
**Date:** March 2026

---